In [1]:
#Import libraries
import pandas as pd
import re

In [2]:
#Load the dataset of csv files and 
postings = pd.read_csv('linkedin_job_postings.csv', usecols=['job_link', 'job_title', 'company', 'job_location', 'search_country', 'job_level'])
skills = pd.read_csv('job_skills.csv')

#View the shape of the datasets
print("Postings shape:", postings.shape)
print("Skills shape:", skills.shape)

Postings shape: (1048575, 6)
Skills shape: (1048575, 2)


In [3]:
# Check for unique job links in both datasets and their overlap
postings_links = set(postings['job_link'])
skills_links = set(skills['job_link'])

print("Unique job_links in postings:", len(postings_links))
print("Unique job_links in skills:", len(skills_links))

overlap = postings_links & skills_links
print("Overlap count:", len(overlap))

Unique job_links in postings: 1048575
Unique job_links in skills: 1048575
Overlap count: 928926


In [4]:
# Merge on job_link (inner join keeps only postings that have both title AND skills data)
df = pd.merge(postings, skills, on='job_link', how='inner')
print("Merged shape:", df.shape)

# Define IT-related job title keywords
it_keywords = [
    'software engineer', 'software developer', 'full stack', 'backend developer', 'frontend developer', 'web developer',
    'data analyst', 'data scientist', 'data engineer', 'business intelligence', 'bi analyst',
    'network engineer', 'network admin', 'network administrator', 'network analyst', 'network automation',
    'system admin', 'systems admin', 'sysadmin', 'infrastructure engineer', 'cloud engineer',
    'quality assurance', 'qa engineer', 'qa tester', 'qa analyst', 'qa manager', 'test engineer',
    'security engineer', 'cybersecurity', 'security analyst',
    'devops', 'site reliability', 'sre',
    'database admin', 'dba',
    'help desk', 'it support', 'desktop support', 'it technician',
    'it manager', 'it analyst', 'application support',
    'ux designer', 'ui designer', 'machine learning engineer', 'ai engineer'
]

pattern = '|'.join(it_keywords)
it_mask = df['job_title'].str.contains(pattern, case=False, na=False, regex=True)
it_df = df[it_mask].copy()

#Eliminate non-IT related job titles
exclude_terms = ['nurse', 'nursing', 'therapist', 'physician', 'dental', 'clinical', 'patient care', 'Unit Manager RN', 'Food', 'Fire', 'Flight']
exclude_mask = it_df['job_title'].str.contains('|'.join(exclude_terms), case=False, na=False)
it_df = it_df[~exclude_mask]

#print the IT-related postions out of all the postings
print("Total IT-related postings found:", len(it_df))
print("Top 30 IT job titles by frequency:")
print(it_df['job_title'].value_counts().head(30))

Merged shape: (928926, 7)
Total IT-related postings found: 24315
Top 30 IT job titles by frequency:
job_title
Senior Software Engineer                669
Quality Assurance Manager               285
Software Engineer                       281
Senior Network Engineer                 249
Audit Manager                           230
Senior Data Engineer                    225
Network Engineer                        217
Senior Embedded Software Engineer       203
Senior Data Analyst                     152
Data Engineer                           144
Data Analyst                            125
Senior Software Developer               120
Lead Software Engineer                  116
Systems Administrator                   100
Sr. Software Engineer                   100
Lead Data Engineer                       96
Data Scientist                           94
Senior DevOps Engineer                   94
Network Automation Engineer              94
Senior DDI Network Engineer              91
Test Engin

In [5]:
#Categories count
def categorize_role(title):
    title_lower = title.lower()
    if 'devops' in title_lower or 'site reliability' in title_lower or re.search(r'\bsre\b', title_lower):
        return 'DevOps/SRE'
    elif 'network' in title_lower:
        return 'Network Engineer'
    elif 'security' in title_lower or 'cybersecurity' in title_lower:
        return 'Security Engineer/Analyst'
    elif 'data scientist' in title_lower:
        return 'Data Scientist'
    elif 'data engineer' in title_lower:
        return 'Data Engineer'
    elif 'data analyst' in title_lower or 'business intelligence' in title_lower or re.search(r'\bbi\s*analyst\b', title_lower):
        return 'Data/BI Analyst'
    elif 'quality assurance' in title_lower or re.search(r'\bqa\b', title_lower):
        return 'QA/Quality Assurance'
    elif 'test engineer' in title_lower:
        return 'Test Engineer'
    elif 'database' in title_lower or re.search(r'\bdba\b', title_lower):
        return 'Database Administrator'
    elif 'cloud' in title_lower:
        return 'Cloud Engineer'
    elif 'infrastructure' in title_lower:
        return 'Infrastructure Engineer'
    elif 'system admin' in title_lower or 'systems admin' in title_lower or 'sysadmin' in title_lower:
        return 'Systems Administrator'
    elif 'desktop support' in title_lower or 'help desk' in title_lower or 'it support' in title_lower or 'it technician' in title_lower:
        return 'IT Support/Help Desk'
    elif 'machine learning engineer' in title_lower or 'ai engineer' in title_lower:
        return 'ML/AI Engineer'
    elif 'software engineer' in title_lower or 'software developer' in title_lower or 'full stack' in title_lower or 'backend developer' in title_lower or 'frontend developer' in title_lower or 'web developer' in title_lower:
        return 'Software Engineer/Developer'
    elif re.search(r'\bit\s*manager\b', title_lower):
        return 'IT Manager'
    elif re.search(r'\bit\s*analyst\b', title_lower):
        return 'IT Analyst'
    elif 'ux designer' in title_lower or 'ui designer' in title_lower:
        return 'UX/UI Designer'
    else:
        return 'Other IT'

it_df['job_category'] = it_df['job_title'].apply(categorize_role)

print(it_df['job_category'].value_counts())

job_category
Software Engineer/Developer    7888
Data/BI Analyst                2102
Security Engineer/Analyst      1901
QA/Quality Assurance           1833
Other IT                       1644
Network Engineer               1527
Data Engineer                  1362
Test Engineer                  1024
DevOps/SRE                      896
IT Support/Help Desk            813
Systems Administrator           655
Infrastructure Engineer         545
Data Scientist                  520
Database Administrator          466
ML/AI Engineer                  373
UX/UI Designer                  311
Cloud Engineer                  290
IT Manager                      133
IT Analyst                       32
Name: count, dtype: int64


In [6]:
# Turn the skills text into a list of separate skills for each row
it_df['skills_list'] = it_df['job_skills'].apply(
    lambda x: [skill.strip() for skill in str(x).split(',')] if pd.notna(x) else []
)

# Count how often each skill appears across all postings
from collections import Counter
all_skills = [skill for skills in it_df['skills_list'] for skill in skills]
skill_counts = Counter(all_skills)

# Show the top 40 most common skills
print("Top 40 most common skills:")
for skill, count in skill_counts.most_common(40):
    print(f"{skill}: {count}")

Top 40 most common skills:
Python: 7470
SQL: 5505
Communication: 4566
Java: 4478
AWS: 4083
Kubernetes: 2616
Teamwork: 2584
Agile: 2495
Troubleshooting: 2469
Docker: 2458
Linux: 2408
JavaScript: 2399
Software Engineering: 2292
Problem Solving: 2090
Data Analysis: 2056
Leadership: 2024
Collaboration: 2006
DevOps: 1928
Azure: 1918
Computer Science: 1873
Software Development: 1805
C++: 1792
Git: 1658
Project Management: 1566
Go: 1508
Communication Skills: 1502
Machine Learning: 1482
C#: 1398
Communication skills: 1349
Bachelor's Degree: 1314
Cloud Computing: 1299
Tableau: 1288
React: 1280
Problemsolving: 1248
Jenkins: 1223
NoSQL: 1205
Quality Assurance: 1191
Data Visualization: 1189
Documentation: 1143
Scala: 1132


In [7]:
# Check which AI-specific terms actually appear in the skill list
ai_keywords = [
    'machine learning', 'deep learning', 'tensorflow', 'pytorch', 
    'nlp', 'natural language processing', 'chatgpt', 'llm', 
    'artificial intelligence', 'neural network', 'data science',
    'computer vision', 'generative ai', 'prompt engineering', ' ai ', 'ai/ml'
]

# Check if a single skill mentions any AI keyword
def is_ai_skill(skill):
    skill_lower = ' ' + skill.lower() + ' '
    for keyword in ai_keywords:
        if keyword in skill_lower:
            return True
    return False

# Tag every skill in a posting as AI or Traditional
def tag_skills(skills_list):
    return ['AI' if is_ai_skill(skill) else 'Traditional' for skill in skills_list]

it_df['skills_tagged'] = it_df['skills_list'].apply(tag_skills)

# Mark whether each posting has at least one AI skill
it_df['has_ai_skill'] = it_df['skills_tagged'].apply(lambda tags: 'AI' in tags)

print("Postings with at least 1 AI-related skill:", it_df['has_ai_skill'].sum())
print("Postings with no AI-related skill:", (~it_df['has_ai_skill']).sum())
print("Percent of postings with AI skill:", round(it_df['has_ai_skill'].mean() * 100, 2), "%")

Postings with at least 1 AI-related skill: 3324
Postings with no AI-related skill: 20991
Percent of postings with AI skill: 13.67 %


In [8]:
# Percent of postings with at least 1 AI skill
ai_by_category = it_df.groupby('job_category')['has_ai_skill'].mean().sort_values(ascending=False) * 100

print("Percent of postings with AI skill, by job category:")
print(ai_by_category.round(2))

Percent of postings with AI skill, by job category:
job_category
ML/AI Engineer                 98.93
Data Scientist                 95.38
Data Engineer                  34.65
Data/BI Analyst                23.03
Cloud Engineer                 14.83
Software Engineer/Developer    14.27
Security Engineer/Analyst       6.26
DevOps/SRE                      5.02
Infrastructure Engineer         4.40
UX/UI Designer                  4.18
Database Administrator          3.43
IT Analyst                      3.12
Test Engineer                   2.34
IT Manager                      2.26
Other IT                        2.13
Network Engineer                1.70
Systems Administrator           1.22
QA/Quality Assurance            0.76
IT Support/Help Desk            0.74
Name: has_ai_skill, dtype: float64


In [9]:
# Check how many rows have missing/empty skills_list
missing_skills = it_df['skills_list'].apply(lambda x: len(x) == 0 if isinstance(x, list) else True)
print("Rows with missing/empty skills_list:", missing_skills.sum())
print("Total rows:", len(it_df))

# Remove rows where skills_list is empty
it_df_clean = it_df[it_df['skills_list'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)].copy()

print("Rows before cleaning:", len(it_df))
print("Rows after removing empty skills:", len(it_df_clean))

text_columns = ['job_title', 'company', 'job_location', 'job_skills']

for col in text_columns:
    it_df_clean[col] = it_df_clean[col].astype(str).str.replace('"', '', regex=False)

print("Stray quotes removed from text columns")

Rows with missing/empty skills_list: 3
Total rows: 24315
Rows before cleaning: 24315
Rows after removing empty skills: 24312
Stray quotes removed from text columns


In [13]:
# Save the cleaned data to a CSV file
it_df_clean.to_csv('cleaned_jobs_dataset.csv', index=False)
print("Final Dataset Shape:", it_df_clean.shape)

Final Dataset Shape: (24312, 11)
